# **Introdução ao LangChain**

**Disciplina:** Generative AI & Advanced Analytics

**Instituição:** PUC Minas

**Professor:** Renan Santos Mendes

**Email:** renansantosmendes@gmail.com

---

Este notebook tem como objetivo apresentar de forma pratica os conceitos iniciais do LangChain, uma biblioteca para construcao de aplicacoes baseadas em modelos de linguagem (LLMs).

Serao abordados os seguintes topicos:

1. Configuracao do ambiente e do modelo de linguagem
2. Messages (SystemMessage, HumanMessage e AIMessage)
3. Templates (PromptTemplate e ChatPromptTemplate)
4. Runnables (conceitos basicos)
5. Chains (encadeamento de componentes)


## 1. Configuracao do ambiente

Antes de comecar, precisamos instalar a biblioteca `langchain-openai`, que fornece a integracao entre o LangChain e modelos compativeis com a API da OpenAI.

Nesta aula, usaremos um proxy proprio para acessar o modelo `gpt-4o-mini`, entao nao sera necessario fornecer uma chave de API valida.

A instalacao sera feita utilizando o `uv`, um gerenciador de pacotes mais rapido que o `pip` tradicional.

In [1]:
!uv pip install langchain-openai -q

## 2. Criando o modelo de linguagem

O `ChatOpenAI` e a classe do LangChain responsavel por representar um modelo de chat compativel com a API da OpenAI.

Abaixo, criamos uma instancia do modelo apontando para o proxy da disciplina.

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key="SUA_MATRICULA",
    temperature=0.99,
    base_url="https://pgl-proxy.vercel.app/v1",
)

Podemos testar o modelo enviando uma pergunta simples diretamente como texto.

In [5]:
response = llm.invoke("me explique o que é uma função em python usando apenas uma frase")

In [6]:
response

AIMessage(content='Uma função em Python é um bloco de código reutilizável que executa uma tarefa específica e pode ser chamado com argumentos para processar dados e retornar um resultado.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 20, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c980d3075f', 'id': 'chatcmpl-EC8iMgTvU8otASFXK1AT9oGxFHFv6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ff766-d764-74e3-bb61-5c1b46469189-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 33, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_r

In [ ]:
dialog = """
**Atendente (Mercado Livre):**
Olá, boa tarde! Meu nome é Juliana e vou acompanhar seu atendimento hoje. Em que posso ajudar?

**Cliente:**
Boa tarde. Estou extremamente insatisfeito. Comprei um notebook de aproximadamente R$ 8.500 e o sistema informa que foi entregue ontem às 15h42, mas eu não recebi absolutamente nada.

**Atendente:**
Sinto muito pelo ocorrido. Vou verificar todas as informações do seu pedido. Poderia confirmar o número do pedido, por favor?

**Cliente:**
Claro. É o pedido #MLB-845973221.

**Atendente:**
Obrigada. Localizei aqui sua compra. Segundo a transportadora, a entrega foi realizada ontem às 15h42 e consta uma assinatura como comprovante.

**Cliente:**
Pois é justamente esse o problema. Eu estava trabalhando presencialmente naquele horário. Moro em condomínio com portaria e ninguém recebeu encomenda nenhuma para mim.

**Atendente:**
Entendo. Vou consultar o comprovante de entrega.

*(alguns segundos depois)*

Vejo que existe uma assinatura registrada, porém ela está ilegível. Também consta apenas o primeiro nome "Carlos". Esse nome corresponde a algum morador ou porteiro do condomínio?

**Cliente:**
Não. Não existe nenhum porteiro chamado Carlos. Inclusive já conversei com a administração do condomínio e eles verificaram o livro de encomendas. Não há nenhum registro dessa entrega.

**Atendente:**
Entendi. Você chegou a verificar se algum vizinho recebeu o pacote por engano?

**Cliente:**
Sim. Falei com praticamente todos do meu andar e dos apartamentos vizinhos. Ninguém recebeu nada.

**Atendente:**
Obrigado pelas informações. Vou abrir uma investigação junto à transportadora. Enquanto isso, gostaria de confirmar alguns dados.

O endereço cadastrado é:

Rua das Palmeiras, 1250, Bloco B, Apto 804.

Está correto?

**Cliente:**
Sim. O endereço está correto e já recebi dezenas de compras nesse mesmo endereço.

**Atendente:**
Perfeito.

Vejo aqui que a entrega foi realizada utilizando geolocalização do entregador. Entretanto, a coordenada GPS apresenta uma diferença aproximada de 450 metros do endereço cadastrado.

**Cliente:**
Então provavelmente entregaram no lugar errado.

**Atendente:**
É uma possibilidade que precisamos investigar.

Além disso, o sistema informa que houve apenas uma tentativa de entrega. Você recebeu alguma ligação, mensagem ou contato do entregador?

**Cliente:**
Não recebi nenhuma ligação.

**Atendente:**
Tudo bem.

Existe outro detalhe importante: o vendedor informou que o produto possui número de série registrado e seguro durante o transporte.

**Cliente:**
Ótimo, mas isso não resolve meu problema. Eu paguei pelo notebook e continuo sem ele.

**Atendente:**
Compreendo totalmente sua preocupação.

Para darmos continuidade, precisarei solicitar algumas evidências para anexar ao processo:

* declaração do condomínio informando que a encomenda não foi registrada;
* imagens das câmeras da portaria entre 15h20 e 16h00, caso disponíveis;
* documento comprovando que você estava em outro local no momento da suposta entrega (caso possua).

**Cliente:**
Tenho tudo isso.

Meu condomínio já emitiu uma declaração.

Também tenho as imagens das câmeras mostrando que nenhuma transportadora entrou nesse horário.

Além disso, consigo comprovar que bati ponto na empresa às 15h31 e saí apenas às 18h.

**Atendente:**
Essas informações serão muito importantes para a análise.

Vou anexar toda a documentação ao protocolo.

**Cliente:**
Quanto tempo isso demora?

**Atendente:**
Normalmente entre 3 e 5 dias úteis.

Entretanto, devido ao valor elevado da compra, o caso será encaminhado para nossa equipe especializada em prevenção a fraudes e auditoria logística.

**Cliente:**
E durante esse período eu fico sem dinheiro e sem notebook?

**Atendente:**
Entendo sua preocupação.

Dependendo do resultado da auditoria preliminar, podemos antecipar o reembolso antes mesmo da conclusão da investigação com a transportadora.

Ainda não consigo garantir isso, mas vou registrar a urgência.

**Cliente:**
O vendedor está me culpando, dizendo que a entrega foi concluída corretamente.

**Atendente:**
Nesse momento não estamos atribuindo responsabilidade a nenhuma das partes.

Nossa equipe irá confrontar:

* GPS do entregador;
* histórico da rota;
* fotografia da entrega, se existir;
* assinatura registrada;
* dados da transportadora;
* imagens fornecidas pelo condomínio;
* documentos enviados por você.

**Cliente:**
Existe fotografia da entrega?

**Atendente:**
No momento não.

O entregador não anexou fotografia, apenas a assinatura digital.

Esse fator também será considerado durante a investigação.

**Cliente:**
Se confirmarem erro da transportadora, recebo outro notebook ou meu dinheiro?

**Atendente:**
Você poderá optar pelo reembolso integral ou, caso ainda haja disponibilidade de estoque e concordância do vendedor, pelo envio de um novo produto.

**Cliente:**
Prefiro o reembolso. Já perdi a confiança nessa entrega.

**Atendente:**
Sem problemas.

Vou registrar sua preferência.

Protocolo: 2026-45892173.

Caso a investigação confirme inconsistências na entrega, o estorno será realizado automaticamente pelo mesmo meio de pagamento utilizado na compra.

**Cliente:**
Preciso fazer mais alguma coisa?

**Atendente:**
Neste momento, apenas envie os documentos solicitados através deste atendimento.

Assim que forem recebidos, sua solicitação seguirá para análise prioritária.

Você também receberá atualizações automáticas sempre que houver movimentação no caso.

**Cliente:**
Tudo bem. Vou enviar agora.

Espero realmente que isso seja resolvido.

**Atendente:**
Pode ficar tranquilo. Acompanharei o processo até a conclusão e faremos todo o possível para resolver a situação da forma mais rápida e justa.

Existe mais alguma dúvida em que eu possa ajudar hoje?

**Cliente:**
Não. Obrigado pelo atendimento.

**Atendente:**
Eu que agradeço pelo contato. Tenha uma boa tarde.

"""

In [ ]:
%%time
summary = llm.invoke(f"Resuma o diálogo a seguir extraindo os principais pontos do atendimento ao cliente: \n\n {dialog}")

CPU times: user 6.33 ms, sys: 1.24 ms, total: 7.57 ms
Wall time: 8.61 s


In [ ]:
summary

AIMessage(content='**Resumo do Atendimento ao Cliente:**\n\n- **Insatisfação Inicial:** O cliente expressa insatisfação com a entrega de um notebook de R$ 8.500, que, segundo o sistema, foi entregue, mas o cliente não recebeu.\n  \n- **Verificação de Informações:** A atendente, Juliana, confirma o número do pedido e identifica que a entrega foi registrada com assinatura ilegível e um nome que não corresponde a ninguém conhecido pelo cliente.\n\n- **Investigação da Situação:** A atendente sugere verificar se algum vizinho recebeu o pacote por engano, mas o cliente confirma que ninguém no condomínio recebeu a entrega. O endereço cadastrado está correto, mas a entrega foi feita a 450 metros do local registrado.\n\n- **Documentação Necessária:** Para continuar a investigação, a atendente solicita evidências do cliente, como uma declaração do condomínio e imagens das câmeras de segurança, bem como comprovantes de que o cliente estava em outro local no momento da entrega.\n\n- **Prazos:** O 

In [ ]:
print(summary.content)

**Resumo do Atendimento ao Cliente:**

- **Insatisfação Inicial:** O cliente expressa insatisfação com a entrega de um notebook de R$ 8.500, que, segundo o sistema, foi entregue, mas o cliente não recebeu.
  
- **Verificação de Informações:** A atendente, Juliana, confirma o número do pedido e identifica que a entrega foi registrada com assinatura ilegível e um nome que não corresponde a ninguém conhecido pelo cliente.

- **Investigação da Situação:** A atendente sugere verificar se algum vizinho recebeu o pacote por engano, mas o cliente confirma que ninguém no condomínio recebeu a entrega. O endereço cadastrado está correto, mas a entrega foi feita a 450 metros do local registrado.

- **Documentação Necessária:** Para continuar a investigação, a atendente solicita evidências do cliente, como uma declaração do condomínio e imagens das câmeras de segurança, bem como comprovantes de que o cliente estava em outro local no momento da entrega.

- **Prazos:** O cliente pergunta sobre o temp

In [ ]:
print(response.content)

Uma função em Python é um bloco de código reutilizável que realiza uma tarefa específica, podendo receber entradas (argumentos) e retornar um resultado.


## 3. Messages

Ao trabalhar com modelos de chat, a comunicacao e organizada em **mensagens**, cada uma com um papel especifico dentro da conversa. As tres principais mensagens do LangChain sao:

- **SystemMessage**: define o comportamento, o tom ou as regras que o modelo deve seguir durante toda a conversa. E como se fosse uma instrucao dada ao modelo antes do dialogo comecar.
- **HumanMessage**: representa a fala do usuario, ou seja, a pergunta ou solicitacao feita ao modelo.
- **AIMessage**: representa a resposta gerada pelo modelo. Tambem pode ser usada para simular respostas anteriores do modelo em um historico de conversa.

Vamos importar essas classes e construir uma conversa simples.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = ...

response = llm.invoke(messages)
print(response.content)

Podemos tambem simular um historico de conversa, incluindo uma resposta anterior do modelo (`AIMessage`) antes de fazer uma nova pergunta. Isso ajuda o modelo a manter contexto sobre o que ja foi dito.

In [ ]:
conversation_history = ...

response = llm.invoke(conversation_history)
print(response.content)

## 4. Templates

Em aplicacoes reais, raramente escrevemos prompts fixos: normalmente queremos reaproveitar uma mesma estrutura de prompt, alterando apenas alguns valores. Para isso, o LangChain oferece os **templates**.

- **PromptTemplate**: usado para criar um template de texto simples, com variaveis que serao preenchidas dinamicamente.
- **ChatPromptTemplate**: usado para criar um template composto por varias mensagens (system, human, etc.), tambem com variaveis dinamicas.

Vamos comecar com o `PromptTemplate`.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = ...

formatted_prompt = prompt_template.format(concept="loop")
print(formatted_prompt)

Agora vamos usar o `ChatPromptTemplate`, que permite definir varias mensagens dentro do mesmo template, cada uma com seu papel (system, human, etc.).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt_template = ...

formatted_messages = chat_prompt_template.format_messages(concept="funcao")
for message in formatted_messages:
    print(message)

## 5. Runnables

Todo componente do LangChain que pode ser executado (um modelo, um template, uma funcao de transformacao de dados, entre outros) implementa a interface `Runnable`.

Um `Runnable` expoe metodos padronizados, como:

- `invoke`: executa o componente com uma unica entrada
- `batch`: executa o componente com varias entradas de uma vez
- `stream`: executa o componente retornando a resposta em partes (streaming)

Essa padronizacao e o que permite combinar diferentes componentes entre si, formando as **chains**, que veremos a seguir.

Abaixo, um exemplo simples mostrando que tanto o `ChatPromptTemplate` quanto o `llm` sao `Runnables`, pois ambos possuem o metodo `invoke`.

In [ ]:
print(hasattr(chat_prompt_template, "invoke"))
print(hasattr(llm, "invoke"))

## 6. Chains

Uma **chain** e o encadeamento de dois ou mais `Runnables`, de forma que a saida de um componente seja usada como entrada do proximo.

No LangChain, esse encadeamento e feito de maneira declarativa usando o operador `|` (pipe), conhecido como LCEL (LangChain Expression Language).

Abaixo, vamos criar uma chain simples que:

1. Recebe um conceito
2. Formata o prompt usando o `ChatPromptTemplate`
3. Envia o prompt formatado para o modelo `llm`

In [ ]:
chain = ...

response = chain.invoke({"concept": "recursao"})
print(response.content)

Podemos tambem adicionar mais um passo a chain, por exemplo, extraindo apenas o texto da resposta usando o `StrOutputParser`, que converte a saida do modelo (um `AIMessage`) em uma string simples.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain_with_parser = ...

result = chain_with_parser.invoke({"concept": "lista encadeada"})
print(result)
print(type(result))

## 7. Runnables avançados: RunnableParallel e RunnableBranch

Alem do encadeamento simples com o operador `|`, o LangChain oferece componentes que permitem organizar a execucao dos `Runnables` de formas mais elaboradas. Dois exemplos bastante uteis sao:

- **RunnableParallel**: executa varios `Runnables` ao mesmo tempo, usando a mesma entrada, e retorna um dicionario com o resultado de cada um. E util quando queremos, por exemplo, gerar respostas diferentes para o mesmo conceito (uma explicacao e um exemplo de codigo, ao mesmo tempo).
- **RunnableBranch**: permite definir diferentes caminhos de execucao (chains diferentes) de acordo com uma condicao aplicada sobre a entrada. Funciona como uma estrutura de `if / elif / else` para `Runnables`.

Vamos ver um exemplo simples de cada um.

### 7.1 RunnableParallel

No exemplo abaixo, vamos executar duas chains diferentes ao mesmo tempo, a partir do mesmo conceito de entrada:

- uma chain que gera uma explicacao teorica
- uma chain que gera um exemplo de codigo em Python

O resultado sera um dicionario contendo as duas respostas.

In [ ]:
from langchain_core.runnables import RunnableParallel

explanation_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

code_example_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Escreva um exemplo curto de codigo em Python sobre {concept}."),
])

explanation_chain = explanation_prompt | llm | StrOutputParser()
code_example_chain = code_example_prompt | llm | StrOutputParser()

parallel_chain = ...

parallel_result = parallel_chain.invoke({"concept": "list comprehension"})
print(parallel_result["explanation"])
print("---")
print(parallel_result["code_example"])

### 7.2 RunnableBranch

No exemplo abaixo, vamos criar duas chains diferentes: uma especializada em conceitos de programacao e outra especializada em conceitos de matematica. O `RunnableBranch` sera responsavel por escolher qual chain executar, de acordo com o valor do campo `topic` presente na entrada.

Cada condicao do `RunnableBranch` e uma tupla no formato `(funcao_condicao, chain)`. A ultima entrada, sem condicao, funciona como o caminho padrao (`else`).

In [ ]:
from langchain_core.runnables import RunnableBranch

programming_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

math_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de matematica."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

default_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um assistente generalista."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

programming_chain = programming_prompt | llm | StrOutputParser()
math_chain = math_prompt | llm | StrOutputParser()
default_chain = default_prompt | llm | StrOutputParser()

branch_chain =...

programming_result = branch_chain.invoke({"topic": "programming", "concept": "recursao"})
print(programming_result)
print("---")
math_result = branch_chain.invoke({"topic": "math", "concept": "derivada"})
print(math_result)